# VC0 · Demo de la videoconferencia de bienvenida
## "Un clúster en vuestro navegador"

**Esto NO es la actividad evaluable y no se entrega.** Es la demostración de la
videoconferencia de presentación del módulo. Aquí puedes ejecutar, romper y experimentar sin
consecuencias.

**No hay que preparar nada.** No genera datos, no lee ficheros y no necesita ninguna
instalación: todo lo que usa ya viene en el devcontainer del curso. Es a propósito — es la
demo con menos cosas que puedan fallar en directo de todo el módulo.

**Qué demuestra, en tres gestos:**

1. El entorno del curso está listo, sin haber instalado nada.
2. Spark procesa decenas de millones de filas en segundos, dentro de un navegador.
3. Lo que Spark hace por dentro **se puede mirar**: la Spark UI, en el puerto 4040.

> **Antes del directo:** ejecuta el cuaderno entero una vez, **anota tus tiempos** y borra las
> salidas. Los números de abajo son de referencia; el que digas en clase tiene que ser el que
> salga en tu pantalla.

---
## 1 · El entorno

Esta celda comprueba lo mismo que el devcontainer imprime al arrancar. Si sale `Entorno OK`,
todo lo demás funciona.

In [ ]:
import pyspark, deltalake, duckdb, dbt.version, prometheus_client

print("Entorno OK · PySpark", pyspark.__version__)
print("  Delta Lake", deltalake.__version__, "· DuckDB", duckdb.__version__)

---
## 2 · Arranca el motor

Levantar Spark **tarda**: hay una JVM que iniciar. Pasa una sola vez, y por eso esta celda se
ejecuta **antes** de empezar a hablar del número que vamos a medir.

La última línea calienta la máquina virtual con un trabajo mínimo, para que la medición de
después sea del trabajo de verdad y no del arranque.

In [ ]:
import time
from pyspark.sql import SparkSession, functions as F

t0 = time.time()
spark = (SparkSession.builder
         .master("local[*]")          # esta máquina, todos sus núcleos
         .appName("bienvenida")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(f"Arrancar la sesión de Spark: {time.time() - t0:.1f} s")

t0 = time.time()
spark.range(1).count()               # calentamiento de la JVM
print(f"Calentamiento:               {time.time() - t0:.1f} s")

print("\nSpark UI:", spark.sparkContext.uiWebUrl)

> **Abre la Spark UI** ahora, en otra pestaña: el enlace de arriba, o en Codespaces la pestaña
> *Ports* → puerto **4040**. Déjala abierta. La vamos a mirar dentro de un momento.

---
## 3 · Cincuenta millones de filas

Genera 50 millones de números, los agrupa en siete cubos y cuenta cuántos hay en cada uno.
El resultado da igual. Lo que importa es **cuánto tarda** y **dónde está corriendo**.

In [ ]:
N = 50_000_000     # si en tu máquina va lento, baja a 20_000_000

t0 = time.time()
(spark.range(N)
      .groupBy((F.col("id") % 7).alias("resto"))
      .count()
      .orderBy("resto")
      .show())
print(f"{N:,} filas agrupadas en {time.time() - t0:.1f} s")

---
## 4 · Lo que ha pasado por dentro

Vuelve a la **Spark UI** y entra en el *job* que acaba de aparecer.

Verás que Spark **no lo ha hecho de una sola pasada: lo ha partido en dos etapas**. Primero
cuenta por separado en cada trozo de datos; después junta esos resultados parciales. Esa línea
que separa las dos etapas es el punto donde los datos se mueven de sitio, y es la operación más
cara de Spark.

**Por qué son dos, y cómo se consigue que sean menos, es el NF2.** Hoy basta con saber que
esto no es una caja negra: se puede abrir y mirar.

---

### Para cerrar

```python
spark.stop()
```

Y acuérdate de **detener el Codespace** cuando termines.

In [ ]:
spark.stop()
print("Sesión cerrada.")